**YOLOv8** modeli eğitmek için gerekli kütüphane. **Ultralytics** görüntüde nesne tanıma yapan python kütüphanesidir.

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.6 MB/s eta 0:00:00


**Content** (Colab'ın verdiği geçici çalışma klasörü) içeriğini listeler.

In [ ]:
!ls /content

sample_data


**Drive**'ın Colab'a bağlanıp bağlanmadığıını kontrol eder.

In [ ]:
!ls /content/drive

ls: cannot access '/content/drive': No such file or directory


**Drive'ı** Colab'a bağlayarak dosyalara erişim sağlar.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Drive'ın** içeriğini gösterir.

In [ ]:
!ls /content/drive/MyDrive

 Classroom  'Colab Notebooks'


**Roboflow** kütüphanesini yükleme

In [ ]:
!pip install roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 57.5 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="hIpzPluVfipJW8cQLEGe")
project = rf.workspace("sky-zfxvm").project("coco-yrx1j")
version = project.version(1)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to COCO-1 in yolov8:: 100%|██████████| 19812/19812 [00:03<00:00, 6444.44it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


**Parametreler** ayarlanır ve YOLOv8 modeli, eklenen veri seti ile eğitilir.

In [ ]:
!yolo train model=yolov8m.pt data=/content/COCO-1/data.yaml epochs=45 imgsz=640 batch=16

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/COCO-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=45, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0,

In [ ]:
!pip install gradio ultralytics zipfile36

In [ ]:
!pip install ultralytics

In [ ]:
import gradio as gr
import zipfile
import os
import shutil
from ultralytics import YOLO

# ZIP dosyasını kontrol eder ve doğrular.
def train_with_dataset(zip_file, epochs, batch, imgsz):

    base_dir = "/content/uploaded_dataset"
    if os.path.exists(base_dir):
        shutil.rmtree(base_dir)
    os.makedirs(base_dir, exist_ok=True)

    # ZIP dosyasını açar.
    with zipfile.ZipFile(zip_file.name, 'r') as zip_ref:
        zip_ref.extractall(base_dir)

    dataset_type = None
    data_yaml_path = None
    dataset_root = None

    # Veri setinin tipini kontrol eder.
    # data.yaml içeriyorsa, veri seti YOLO formatındadır ve model eğitimi başlatılır.
    for root, dirs, files in os.walk(base_dir):
        if "data.yaml" in files:
            dataset_type = "yolo"
            data_yaml_path = os.path.join(root, "data.yaml")
            dataset_root = root
            break

        # COCO formatına ait annotations.json  veya
        # VOC formatına ait .xml dosyaları olup olmadığını kontrol eder.
        if "annotations.json" in files:
            dataset_type = "coco"
            break

        if any(f.endswith(".xml") for f in files):
            dataset_type = "voc"
            break

    # COCO veya VOC formatlarından biri algılanırsa ekrana uyarı yazdırılır.
    if dataset_type in ["coco", "voc"]:
        return (
            "⚠️ COCO/VOC formatlı veri seti algılandı. "
            "Bu arayüz yalnızca YOLO formatındaki "
            "(data.yaml, images/labels) veri setleriyle çalışır",
            None
        )

    # YOLO, COCO ve VOC formatlarından hiçbiri bulunamazsa ekrana uyarı yazdırılır.
    if dataset_type is None:
        return "❌ Desteklenmeyen dataset formatı", None



    # train / valid klasörünün yapısı kontrol edilir.
    if dataset_type == "yolo" and data_yaml_path:

        train_images = os.path.join(dataset_root, "train/images")
        train_labels = os.path.join(dataset_root, "train/labels")

        val_images = os.path.join(dataset_root, "valid/images")
        val_labels = os.path.join(dataset_root, "valid/labels")

        alt_val_images = os.path.join(dataset_root, "val/images")
        alt_val_labels = os.path.join(dataset_root, "val/labels")


        # Klasörde eksik veya hata olduğu tespit edilirse ekrana uyarı yazdırılır.
        if not (
            os.path.exists(train_images) and os.path.exists(train_labels) and
            (os.path.exists(val_images and val_labels) or os.path.exists(alt_val_images and alt_val_labels))
        ):
            return "❌ train/valid klasör yapısı eksik veya hatalı", None


        try:
            # YOLO Modelini Başlat.
            model = YOLO("yolov8m.pt")

            # Modeli Eğit.
            results = model.train(
                data=data_yaml_path,
                epochs=int(epochs),
                batch=int(batch),
                imgsz=int(imgsz),
                name="custom_yolo_model"
            )

            # Eğitim biter ve dosyalar hazır olur.
            best_model_path = os.path.join(results.save_dir, "weights", "best.pt")

            return f"✅ Eğitim Tamamlandı! Model: {best_model_path}", best_model_path

        except Exception as e:
            return f"❌ Eğitim sırasında hata oluştu: {str(e)}", None

    else:
        # COCO veya VOC geldiyse (YOLOv8 doğrudan data.yaml ister)
        return f"⚠️ {dataset_type} formatı bulundu ancak YOLOv8 eğitimi için 'data.yaml' formatına dönüştürülmesi gerekiyor.", None


# Kullanıcının veri seti (ZIP) yükleyebileceği web arayüzü oluşturulur.
interface = gr.Interface(
    fn=train_with_dataset,
    inputs=[
        gr.File(label="YOLO Veri Seti (ZIP)"),
        gr.Number(value=45, label="Epoch"),
        gr.Number(value=16, label="Batch Size"),
        gr.Number(value=640, label="Image Size")
    ],
    outputs=[
        gr.Text(label="Durum"),
        gr.File(label="Eğitilen Model (best.pt)")
    ],
    title="No-Code YOLOv8 Trainer",
    description=(
        "YOLO formatında, images–labels klasörleri ve data.yaml içeren ZIP veri seti yükleyiniz. "
        "Eksik dosya olması durumunda sistem sizi uyarır. "
        "Eğitim tamamlandığında eğitilen model dosyasını indirebilirsiniz."
    )
)

# YOLOv8 eğitim arayüzünü tarayıcıda çalıştırır.
interface.launch()



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2234e0699f57ec020f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
